In [1]:
import pandas as pd;
import numpy as np;
from SpatialCheck import isSpatialWithinBounds
import geopandas


# DOWNLOAD FILE FIRST: https://syd1.digitaloceanspaces.com/duckgoesmeow/bushfire-data/data.csv
FILE_PATH = "~/Desktop/archive/data.csv"

bushfire_df = pd.read_csv(FILE_PATH, low_memory=False)

columns_of_interest = ['OBJECTID','DISCOVERY_DATE', 'DISCOVERY_TIME', 'NWCG_GENERAL_CAUSE', 'CONT_DATE', 'CONT_TIME', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE' , 'LONGITUDE' , 'STATE' ]
renamed_columns = ["object_id", "discovery_date", "discovery_time", "general_cause", "controlled_date", "controlled_time", "fire_size", "fire_class", "latitude", "longitude", "state", "county"]
map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Undefined',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}



# Let's rename the columns:
bushfire_df = bushfire_df.rename(columns={
                         'OBJECTID':'object_id',
                         'DISCOVERY_DATE':'discovery_date',
                         'DISCOVERY_TIME' : 'discovery_time',
                         'NWCG_GENERAL_CAUSE':'general_cause',
                         'CONT_DATE':'controlled_date',
                         'CONT_TIME':'controlled_time',
                         'FIRE_SIZE' : 'fire_size',
                         'FIRE_SIZE_CLASS' : 'fire_class',
                         'LATITUDE': 'latitude',
                         'LONGITUDE':'longitude',
                         'COUNTY' : 'county',
                         'STATE':'state'}).rename_axis('index_id')

In [2]:
bushfire_df.memory_usage().sum() / 1024 ** 2

np.float64(685.4178619384766)

In [3]:
bushfire_df = bushfire_df[renamed_columns]

In [4]:
bushfire_df['discovery_date'] = pd.to_datetime(arg=bushfire_df['discovery_date']).astype(str)

In [5]:
bushfire_df['discovery_time'] = pd.to_numeric(bushfire_df['discovery_time'], errors='coerce').fillna(0).astype(int).astype(str)

In [6]:
def format_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).zfill(4)
    if time_str == '0000' or time_str == '2400':
        return '00:00'
    elif 0 <= int(time_str) <= 2359:
        return f"{time_str[:2]}:{time_str[2:]}"
    else: 
        return np.nan
    

bushfire_df['discovery_time'] = bushfire_df['discovery_time'].apply(format_time)

In [7]:
bushfire_df['discovery_datetime'] = bushfire_df['discovery_date'] + " " + bushfire_df['discovery_time']

In [8]:
bushfire_df['discovery_datetime'] = pd.to_datetime(arg=bushfire_df['discovery_datetime'], format='%Y-%m-%d %H:%M')

In [9]:
bushfire_df.insert(11, 'discovery_day', bushfire_df['discovery_datetime'].dt.day_name())

In [10]:
bushfire_df['discovery_datetime'] = bushfire_df['discovery_datetime'].dt.strftime('%Y-%m-%d %H:%M')

In [11]:
bushfire_df.drop(columns=['discovery_date', 'discovery_time'], axis=1, inplace=True)

In [12]:
bushfire_df.drop(columns=['county'], axis=1, inplace=True)

In [13]:
map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Missing',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}


bushfire_df['origin'] = bushfire_df['general_cause'].map(map_cause)

In [14]:
cols_to_fill = ['controlled_date', 'controlled_time']
bushfire_df[cols_to_fill] = bushfire_df[cols_to_fill].fillna('unknown')

In [15]:
output_file = './cleaned-bushfire-data.csv'
bushfire_df.to_csv(output_file, index=False, encoding='utf-8')

In [16]:
cleaned_df = pd.read_csv('./cleaned-bushfire-data.csv')

In [17]:
cleaned_df['controlled_time'] = pd.to_numeric(cleaned_df['controlled_time'], errors='coerce').fillna('unknown').astype(str)

In [18]:
# TODO: Test all of the fire_size and see if the match with their corresponding fire_class
fire_size_classification = [{"size_min": 0.1, "size_max": 0.25, "class": "A"}, {"size_min": 0.26, "size_max": 9, "class": "B"}, { "size_min": 9.1, "size_max": 99, "class": "C"}, { "size_min": 100, "size_max": 299, "class": "D"}, { "size_min": 300, "size_max": 999, "class": "E"}, { "size_min": 1000, "size_max": 4999, "class": "F"}, { "size_min": 5000, "size_max": 9999, "class": "G"}]

In [19]:
cleaned_df.drop(columns=['controlled_time', 'controlled_date'], inplace=True)

In [20]:
cleaned_df.drop(cleaned_df.loc[cleaned_df['origin'] == 'Missing'].index, inplace=True)

In [21]:
cleaned_df.drop(cleaned_df.loc[cleaned_df['origin'] == 'Accidental'].index, inplace=True)

In [22]:
cleaned_df.drop(cleaned_df.loc[cleaned_df['origin'] == 'Criminal'].index, inplace=True)

In [25]:
# isSpatialWithinBounds(cleaned_df['latitude'], cleaned_df['longitude'], "CA")

cleaned_df.head()

# match by satate first, then narrow it down by matching the lat and long
# draw graph map of most bushfire-dense states
# it's to uncover patterns with the bushfires
# TODO: Gather all of the tempretures within the state and see how much they vary between different 

,object_id,general_cause,fire_size,fire_class,latitude,longitude,state,discovery_day,discovery_datetime,origin
1,2,Natural,0.25,A,38.933056,-120.404444,CA,Wednesday,2004-05-12 08:45,Natural
3,4,Natural,0.10,A,38.559167,-119.913333,CA,Monday,2004-06-28 16:00,Natural
4,5,Natural,0.10,A,38.559167,-119.933056,CA,Monday,2004-06-28 16:00,Natural
5,6,Natural,0.10,A,38.635278,-120.103611,CA,Wednesday,2004-06-30 18:00,Natural
6,7,Natural,0.10,A,38.688333,-120.153333,CA,Thursday,2004-07-01 18:00,Natural
